# Extração Scopus (pyscopus) — Produção Bibliográfica

Notebook original (`Quick-Start.ipynb`) era apenas a demonstração padrão da
biblioteca [`pyscopus`](http://zhiyzuo.github.io/python-scopus/) (wrapper da
[API Scopus da Elsevier](https://dev.elsevier.com/)), com exemplos soltos de
busca geral, busca por autor, recuperação de resumo, etc.

Esta versão reaproveita essas mesmas chamadas, mas em forma de **pipeline de
extração**: dada uma lista de pessoas (com `id_lattes` e `scopus_author_id`),
para cada uma buscamos todas as publicações indexadas na Scopus e produzimos
um DataFrame **no mesmo formato de saída do `analyse.ipynb`**:

- `df_artigos_periodico_scopus` → mesmo schema de `df_artigos_final`

A Scopus indexa majoritariamente periódicos (e alguns proceedings de
conferência classificados como `aggregationType == 'Conference Proceeding'`);
tratamos os dois casos, mas o `analyse.ipynb` já faz seu próprio cruzamento de
periódicos com a base de percentil Scopus — aqui geramos a produção **bruta**
do pesquisador na Scopus, pronta para concatenar.

## 1. Instalação e Configuração

In [ ]:
%pip install pyscopus python-dotenv

In [1]:
import time
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from pyscopus import Scopus

load_dotenv()

# Sua API Key da Elsevier (https://dev.elsevier.com/) — registre uma chave e
# guarde em uma variável de ambiente SCOPUS_API_KEY (ex.: em um arquivo .env)
API_KEY = os.getenv('SCOPUS_API_KEY')

scopus = Scopus(API_KEY)
print("Cliente Scopus inicializado com sucesso!")

ModuleNotFoundError: No module named 'pkg_resources'

## 2. Lista de pessoas a extrair

Cada pessoa precisa ter:

- `id_lattes`: a chave que une todas as fontes (Lattes, ORCID, Scopus) — é o que vai virar a FK no banco.
- `scopus_author_id`: o Author ID da Scopus da pessoa (ex.: `'57189222659'`), usado só para consultar a API.

Substitua a lista de exemplo abaixo pela sua lista real.

In [ ]:
# Exemplo de lista de pessoas. Troque por:
#   df_pessoas_lista = pd.read_csv('lista_pessoas.csv')  # colunas: id_lattes, orcid_id, scopus_author_id
lista_pessoas_scopus = [
    {'id_lattes': '0000000000000001', 'scopus_author_id': '57189222659'},
    # {'id_lattes': '...', 'scopus_author_id': '...'},
]

print(f"Total de pessoas a processar: {len(lista_pessoas_scopus)}")

## 3. Extração das publicações de cada autor

Para cada pessoa usamos `scopus.search_author_publication(scopus_author_id)`,
que devolve um DataFrame com uma linha por publicação (`title`, `cover_date`,
`publication_name`, `scopus_id`, entre outras colunas). Em seguida, para cada
publicação encontrada, chamamos `scopus.retrieve_abstract(scopus_id)` para
complementar com `DOI`, `ISSN` e páginas — os mesmos campos que o
`df_bib_artigos` do Lattes carrega no `analyse.ipynb`.

> **Nota sobre limites da API:** a Scopus impõe limites de requisições por
> segundo/dia conforme seu plano na Elsevier. O `time.sleep` abaixo evita
> estourar o rate limit; ajuste conforme sua cota.

In [ ]:
def extrair_ano(cover_date):
    """Extrai o ano (int) de uma string de data tipo '2018-08-01'."""
    if not cover_date or pd.isna(cover_date):
        return pd.NA
    try:
        return int(str(cover_date)[:4])
    except (ValueError, TypeError):
        return pd.NA


lista_artigos_periodico_scopus = []

for pessoa in lista_pessoas_scopus:
    id_lattes = pessoa['id_lattes']
    scopus_author_id = pessoa['scopus_author_id']

    print(f"Processando Scopus Author ID {scopus_author_id} (id_lattes={id_lattes})...")

    try:
        pub_df = scopus.search_author_publication(scopus_author_id)
    except Exception as exc:
        print(f"  -> Falha ao buscar publicações de {scopus_author_id}: {exc}")
        continue

    if pub_df is None or pub_df.empty:
        print("  -> Nenhuma publicação encontrada.")
        continue

    for _, pub in pub_df.iterrows():
        titulo = pub.get('title', pd.NA)
        revista = pub.get('publication_name', pd.NA)
        ano = extrair_ano(pub.get('cover_date'))
        scopus_id = pub.get('scopus_id', pd.NA)

        doi = pd.NA
        issn = pub.get('issn', pd.NA)

        # Complementa com DOI/ISSN/páginas via retrieve_abstract (1 chamada extra por publicação)
        if pd.notna(scopus_id):
            try:
                pub_info = scopus.retrieve_abstract(str(scopus_id))
                doi = pub_info.get('prism:doi', pd.NA)
                issn = pub_info.get('prism:issn', issn)
            except Exception as exc:
                print(f"  -> Aviso: falha ao detalhar {scopus_id}: {exc}")
            time.sleep(0.2)

        lista_artigos_periodico_scopus.append({
            'id_lattes': id_lattes,
            'titulo_artigo': titulo,
            'titulo_revista_lattes': pd.NA,  # não há contraparte do Lattes aqui
            'ano_pub': ano,
            'doi': doi,
            'autores': pd.NA,  # pyscopus não expõe autores em search_author_publication/retrieve_abstract
            'match_adequado': pd.NA,
            'coautoria_aluno': pd.NA,
            'id_scopus': scopus_id,
            'titulo_revista_scopus': revista,
            'maior_percentil': pd.NA,
            'codigo_area_maior_percentil': pd.NA,
            'area_maior_percentil': pd.NA,
            'issn': issn,
            'computation_area': pd.NA,
        })

    time.sleep(0.2)

print("\nExtração concluída.")
print(f"Artigos extraídos da Scopus: {len(lista_artigos_periodico_scopus)}")

## 4. Consolidação no DataFrame final (mesmo schema do `analyse.ipynb`)

As colunas abaixo replicam exatamente `df_artigos_final`:

`id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub, doi, autores, match_adequado, coautoria_aluno, id_scopus, titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil, area_maior_percentil, issn, computation_area`

Os campos que a Scopus não fornece nesta extração simples (`titulo_revista_lattes`,
`autores`, `maior_percentil`, `area_maior_percentil`, `computation_area`,
`match_adequado`) ficam nulos — o cruzamento de percentil/área já é feito pelo
próprio `analyse.ipynb` (etapa de match com `periodicos_percentil.xlsx`); se
quiser aplicar o mesmo cruzamento aqui, basta reaproveitar aquela lógica
passando `df_artigos_periodico_scopus` no lugar de `df_bib_artigos`.

In [ ]:
colunas_periodico = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area',
]

df_artigos_periodico_scopus = pd.DataFrame(lista_artigos_periodico_scopus, columns=colunas_periodico)

# Mesma tipagem usada no analyse.ipynb para permitir o concat sem surpresas
if not df_artigos_periodico_scopus.empty:
    df_artigos_periodico_scopus['ano_pub'] = pd.to_numeric(df_artigos_periodico_scopus['ano_pub'], errors='coerce').astype('Int64')
    df_artigos_periodico_scopus['id_lattes'] = df_artigos_periodico_scopus['id_lattes'].astype(str)
    if 'titulo_revista_scopus' in df_artigos_periodico_scopus.columns:
        df_artigos_periodico_scopus['titulo_revista_scopus'] = (
            df_artigos_periodico_scopus['titulo_revista_scopus'].astype(str).str.upper().str.strip()
        )

print("=== df_artigos_periodico_scopus ===")
display(df_artigos_periodico_scopus.head())
df_artigos_periodico_scopus.info()

## 5. Próximo passo: unificar com `analyse.ipynb` e `orcid.ipynb`

```python
df_periodicos_unificado = pd.concat(
    [df_artigos_final, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)
```

Esse DataFrame único já está no formato esperado por `tb_artigo_periodico` no
DuckDB (mesmo `INSERT INTO tb_artigo_periodico (...)` que o `analyse.ipynb`
já define), pronto para subir ao banco. Antes de inserir, garanta que toda
pessoa referenciada em `id_lattes` já exista em `tb_professores` (a FK exige isso).

Opcionalmente, é possível remover duplicatas entre as três fontes (ex.: o
mesmo artigo aparecendo no Lattes e na Scopus) usando `doi` como chave —
quando o `doi` é nulo, cair de volta para `(titulo_artigo, id_lattes)`:

```python
df_periodicos_unificado['chave_dedup'] = df_periodicos_unificado['doi'].fillna(
    df_periodicos_unificado['titulo_artigo'].str.upper().str.strip() + '|' + df_periodicos_unificado['id_lattes']
)
df_periodicos_unificado = df_periodicos_unificado.drop_duplicates(subset=['chave_dedup']).drop(columns=['chave_dedup'])
```